In [1]:
# NS
#optiver-ver (for ensemble)

#optiver-ver-nn-modeling/inference-v1
#optiver-ver-rnn-modeling/inference-v1
#optiver-ver-lgb-modeling/inference-v1


import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from scipy.stats import hmean

from sklearn.ensemble import HistGradientBoostingRegressor
import itertools
import pickle
import joblib
from itertools import combinations
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate, GaussianNoise
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras.layers import concatenate,Dropout
import pickle
from tensorflow.keras.models import load_model
import os 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.metrics import MeanAbsoluteError
from tensorflow.keras.callbacks import Callback
import random
from tensorflow.keras.layers import Input, Embedding, Lambda, Reshape, LSTM, Dense, BatchNormalization, Dropout, concatenate
from tensorflow.keras import backend as K
from tensorflow.keras.layers import ZeroPadding1D
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import RepeatVector

from tensorflow.keras.layers import Input, Embedding, Lambda, Reshape, LSTM, Dense, BatchNormalization, Dropout, concatenate
from tensorflow.keras import backend as K
from tensorflow.keras.layers import ZeroPadding1D, Activation
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import RepeatVector
from tensorflow.keras.layers import MaxPooling1D, AveragePooling1D
from tensorflow.keras.layers import Add

#---------------------------------------------------------------------- setup dashboard ------------------------------------------------------------
nn_ep          = 60
nn_lr          = 0.001
nn_bs          = 2**15

run_pipeline        = True
train_models        = True
memory_threshold    = 24576  #24GB

ens_models          = [1.0,1.0,1.0]

#public-validation
# dates_train = [0,390]
# dates_test = [391,480]

#full-inference
dates_train = [0,480]
dates_test = [-1,-1]

num_models ={'lgb':1,'nn':1,'rnn':1} 



train_path = r"C:\Users\cwang\Desktop\Kaggle_optiver\train.csv"
models_path = r"/kaggle/input/optiver-rnn-just-imb-models/"

#---------------------------------------------------------------------- setup dashboard ------------------------------------------------------------


In [2]:
pd.set_option('mode.chained_assignment', None)


def convert_price_cols_float32(df):

    # Columns containing 'price'
    price_columns = [col for col in df.columns if 'price' in col]
    df[price_columns] = df[price_columns].astype('float32')

    # Columns containing 'wap'
    wap_columns = [col for col in df.columns if 'wap' in col]
    df[wap_columns] = df[wap_columns].astype('float32')

    return df

train = pd.read_csv(train_path).drop(['row_id', 'time_id'], axis = 1)
nan_count = train['target'].isna().sum()
print(f"The 'target' column has {nan_count} NaN values.")

target_median = train['target'].median()
train['target'].fillna(target_median, inplace=True)

print(f"converting prices columns to float32 values.")
train = convert_price_cols_float32(train)
# ----------------------------- Reading train data -------------------------

The 'target' column has 88 NaN values.
converting prices columns to float32 values.


C:\Users\cwang\AppData\Local\Temp\ipykernel_16824\241488516.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['target'].fillna(target_median, inplace=True)


In [3]:
#@title functions

def split_by_date(df, dates):

    df_start, df_end = dates
    df = df[(df['date_id'] >= df_start) & (df['date_id'] <=df_end)].reset_index(drop=True)

    return df



def lag_function(df, columns_to_lag, numbers_of_days_to_lag):

    df_indexed = df.set_index(['stock_id', 'seconds_in_bucket', 'date_id'])
    
    for column_to_lag in columns_to_lag:
        for number_days_to_lag in numbers_of_days_to_lag:
            df_indexed[f'lag{number_days_to_lag}_{column_to_lag}'] = df_indexed.groupby(level=['stock_id', 'seconds_in_bucket'])[column_to_lag].shift(number_days_to_lag)
    
    df_indexed.reset_index(inplace=True)
    
    return df_indexed



def create_diff_lagged_features_within_date_revised(df, columns_to_lag, numbers_of_lag):
    df_copy = df.copy()

    # Store the new columns in a list
    new_columns = []

    # Iterate through each specified lag
    for lag in numbers_of_lag:
        # Create lagged dataframe once per lag value
        lagged_df = df.groupby(['stock_id', 'date_id'])[columns_to_lag].shift(periods=lag)
        
        # Iterate through each specified column
        for column in columns_to_lag:
            # Compute the new column
            new_col_name = f'{column}_diff_lag{lag}'
            new_column = df[column] - lagged_df[column]
            
            # Store the new column in the list
            new_columns.append(new_column.rename(new_col_name))

    # Concatenate the original dataframe with the new columns
    result_df = pd.concat([df_copy] + new_columns, axis=1)
    
    return result_df


def create_features_to_start_optimized(df, features_list):

    first_values_df = df.groupby(['stock_id', 'date_id'])[features_list].transform('first')

    for feature in features_list:
        feature_to_start_col_name = f'{feature}_to_start'
        df[feature_to_start_col_name] = df[feature] - first_values_df[feature]

    return df


def compute_imbalances(df_, columns, prefix = ''):
    """Computes the differences and imbalances for pairs of columns and stores them in the DataFrame."""
    df = df_.copy()
    for col1, col2 in combinations(columns, 2):
        
        # Sort the columns lexicographically to ensure consistent ordering
        col1, col2 = sorted([col1, col2])
        
        # Compute imbalance directly without creating a temporary difference column
        total = df[col1] + df[col2]
        imbalance_column_name = f'{col1}_{col2}_imb{prefix}'
        
        # Ensure we don't divide by zero
        df[imbalance_column_name] = (df[col1] - df[col2]).divide(total, fill_value=np.nan)

    return df

def compute_percentage_difference(df, columns, prefix = ''):

    df_copy = df.copy()
    
    # Iterate over all combinations of two different price columns
    for col1, col2 in combinations(columns, 2):
        # Sort the columns lexicographically to ensure consistent ordering
        col1, col2 = sorted([col1, col2])

        # Create a new column name based on the price columns
        new_col_name = f'pct_diff_{col1}_vs_{col2}_{prefix}'

        # Compute the percentage difference
        df_copy[new_col_name] = (df_copy[col1] - df_copy[col2]) / df_copy[col2] * 100

    return df_copy



def create_deviation_within_seconds(df, num_features):
    groupby_cols = ['date_id', 'seconds_in_bucket']
    new_columns = {}  # Dictionary to hold new columns

    for feature in num_features:
        grouped_median = df.groupby(groupby_cols)[feature].transform('median')
        deviation_col_name = f'deviation_from_median_{feature}'
        new_columns[deviation_col_name] = df[feature] - grouped_median

    # Concatenate all new columns at once
    df = pd.concat([df, pd.DataFrame(new_columns)], axis=1)
    return df


def create_cumsum_features(df, columns_to_compute):
    df_copy = df.copy()
    
    # Group by 'stock_id' and 'date_id' for cumulative sum calculation
    grouped = df_copy.groupby(['stock_id', 'date_id'])
    
    # Calculate cumulative sum for each column within each group
    for column in columns_to_compute:
        cumsum_col_name = f'{column}_cumsum'
        df_copy[cumsum_col_name] = grouped[column].cumsum()
    return df_copy


def save_pickle(data, file_path):
 
    # Create the directory if it doesn't exist
    directory = os.path.dirname(file_path)
    if not os.path.exists(directory):
        os.makedirs(directory)

    # Save the pickle file
    with open(file_path, 'wb') as file:
        pickle.dump(data, file)

    print(f"Data saved to {file_path}")
    #example: save_pickle(all_data, 'k8/all_data.pkl')

def load_pickle(file_path):

    # Load and return the data from the pickle file
    if os.path.exists(file_path):
        with open(file_path, 'rb') as file:
            data = pickle.load(file)
        return data
    else:
        raise FileNotFoundError(f"No such file: {file_path}")


In [4]:
#@title global
#---------------------------------- Global based on stock_id --------------------------------------

def aggregated_features_dic(df):
    global_feats = {}

    columns_to_aggregate= ['bid_size','ask_size']
    groupby_cols=['stock_id']

    def q25(x):
        return x.quantile(0.25)

    def q75(x):
        return x.quantile(0.75)

    # Define the aggregations
    aggregations = ['mean', 'median', 'std', 'min', 'max', q25, q75]

    # Prepare a dictionary to hold the aggregated Series


    # Perform aggregation for each column and operation
    for column in columns_to_aggregate:
        for agg in aggregations:
            # Define the aggregation function name
            if callable(agg):
                func_name = agg.__name__
            else:
                func_name = agg
            
            # Perform the aggregation
            agg_series = df.groupby(groupby_cols)[column].agg(agg)
            # Create a new feature name and add it to the dictionary
            new_feature_name = f"{func_name}_{column}"
            global_feats[new_feature_name] = agg_series

    global_feats["median_size"] = df.groupby("stock_id")["bid_size"].median() + df.groupby("stock_id")["ask_size"].median()
    global_feats["std_size"] = df.groupby("stock_id")["bid_size"].std() + df.groupby("stock_id")["ask_size"].std()
    global_feats["ptp_size"] = df.groupby("stock_id")["bid_size"].max() - df.groupby("stock_id")["bid_size"].min()
    global_feats["median_price"] = df.groupby("stock_id")["bid_price"].median() + df.groupby("stock_id")["ask_price"].median()
    global_feats["std_price"] = df.groupby("stock_id")["bid_price"].std() + df.groupby("stock_id")["ask_price"].std()
    global_feats["ptp_price"] = df.groupby("stock_id")["bid_price"].max() - df.groupby("stock_id")["ask_price"].min()


    return global_feats

aggregated_dic = aggregated_features_dic(train)

def map_global(df,dict):
    df_ = df.copy()
    for key, value in dict.items():
        df_[f"global_{key}"] = df_["stock_id"].map(value.to_dict())
    
    return df_


In [5]:
#@title helper functions

def flatten_outliers_y_train(y_train, lower_quantile=0.01, upper_quantile=0.99):

    lower_bound = np.quantile(y_train, lower_quantile)
    upper_bound = np.quantile(y_train, upper_quantile)

    # Cap values below the lower bound and above the upper bound
    y_train_flattened = np.clip(y_train, lower_bound, upper_bound)

    return y_train_flattened


def create_autocorrelation_features(df, columns, lags):

    df_copy = df.copy()
    
    for column in columns:
        for lag in lags:
            lagged_series = df_copy[column].shift(lag)
            df_copy[f'{column}_autocorr_lag{lag}'] = df_copy[column].corrwith(lagged_series)

    return df_copy


def calculate_stat_lag(df, num_lags):

    lags = [f'lag{i}_target' for i in range(1, num_lags + 1)]

    df['target_mean'] = df[lags].mean(axis=1)
    df['target_std_dev'] = df[lags].std(axis=1)
    df['target_variance'] = df[lags].var(axis=1)
    df['target_median'] = df[lags].median(axis=1)
    df['target_range'] = df[lags].max(axis=1) - df[lags].min(axis=1)

    return df


def calculate_stat(df, cols, prefix='prices'):

    df[f'{prefix}_mean'] = df[cols].mean(axis=1)
    df[f'{prefix}_std_dev'] = df[cols].std(axis=1)
    df[f'{prefix}_variance'] = df[cols].var(axis=1)
    df[f'{prefix}_median'] = df[cols].median(axis=1)
    df[f'{prefix}_range'] = df[cols].max(axis=1) - df[cols].min(axis=1)

    return df

In [6]:
def make_predictions(models, X_test,model = 'nn'):
    if model == 'nn':
        all_predictions = [model.predict(X_test, batch_size=16384) for model in models]
    if model == 'lgb' or model == 'xgb' or model == 'cat':
        all_predictions = [model.predict(X_test) for model in models]
    prediction = np.mean(all_predictions, axis=0)
    return prediction

In [7]:
#@title pipeline

raw_cols          = ['imbalance_size','matched_size','bid_size','ask_size','reference_price','far_price','near_price','bid_price','ask_price','wap','imbalance_buy_sell_flag'] 

columns_prices    = ['reference_price','far_price','near_price','bid_price','ask_price','wap']
columns_4prices   = ['reference_price','bid_price','ask_price','wap']

columns_sizes     = ['imbalance_size','matched_size','bid_size','ask_size']
columns_flag      = ['imbalance_buy_sell_flag'] 


diff_lags           = [1, 2, 3, 6, 12, 18, 24]
diff_lags_extra     = [30, 36, 42, 48]

num_of_target_lags  = 12
target_lags         = list(range(1,num_of_target_lags+1))


def feature_engineering(df):
    
    df = df.copy()

    df['spread_eng']                            = df['ask_price'] - df['bid_price'] 
    df['volume_eng']                            = df['bid_size'] + df['ask_size']  
    df['volumne_imbalance_eng']                 = df['bid_size'] - df['ask_size']  
    
    df['imbalance_ratio']                       = df['imbalance_size'] / df['matched_size']  #(RM) Good

    df['price_spread_near_far']                 = df['near_price'] - df['far_price']   #RM (debatable)
    df['price_wap_difference_eng']              = df['reference_price'] - df['wap']    

    df['weighted_imbalance_eng']                = df['imbalance_size'] * df['imbalance_buy_sell_flag'] #very important


    df['bid_ask_ratio']                         = df['bid_size'] / df['ask_size'] #(RM) neutral
    df['imbalance_to_bid_ratio_eng']            = df['imbalance_size'] / df['bid_size']
    df['imbalance_to_ask_ratio_eng']            = df['imbalance_size'] / df['ask_size']
    df['matched_size_to_total_size_ratio_eng']  = df['matched_size'] / (df['bid_size'] + df['ask_size'])


    return df


def feature_pipeline(df):
    
    if df.empty:
        return pd.DataFrame()
        
    #--------------- connected ------------
    df = feature_engineering(df)
    df = compute_imbalances(df, columns_sizes,prefix='_sz_')
    df = compute_imbalances(df, columns_prices,prefix = '_pr_')



    eng_features       = [feature for feature in df.columns if "_eng" in feature]
    imb_features_all   = [feature for feature in df.columns if "_imb_" in feature]
    imb_features_price = [feature for feature in df.columns if "_pr_" in feature]
    imb_features_size  = [feature for feature in df.columns if "_sz_" in feature]

    #--------------- connected ------------


    diff_lag_cols = raw_cols + eng_features
    print(f"diff lagging {len(diff_lag_cols)} columns for {len(diff_lags)} lags.")
    df = create_diff_lagged_features_within_date_revised(df,diff_lag_cols,diff_lags)

    cumsum_columns = columns_sizes + imb_features_size + eng_features 
    print(f"cumsum for {len(cumsum_columns)} cols.")
    df = create_cumsum_features(df, cumsum_columns)

    deviation_cols = raw_cols + eng_features + imb_features_size #+ imb_features_price
    print(f"deviation {len(deviation_cols)} columns within seconds.")
    df = create_deviation_within_seconds(df,deviation_cols)

    print(f"lagging target column for {len(target_lags)} lags.")
    df = lag_function(df, ['target'], target_lags)

    df = map_global(df,aggregated_dic)

    df = calculate_stat_lag(df, num_lags=num_of_target_lags)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    print("Done...")
    
    return df

In [8]:
#@title runnig pipeline

excluded_columns = ['row_id', 'date_id', 'time_id', 'target','stock_return']  

train_eng = feature_pipeline(train)

excluded_columns = excluded_columns + ['stock_id']
features = [col for col in train_eng.columns if col not in excluded_columns]
categorical_features =  ['seconds_in_bucket']
numerical_features = [feat for feat in features if feat not in categorical_features]
print("we have {} numerical and {} categorical".format(len(numerical_features),len(categorical_features)))


scaler = StandardScaler()
medians = train_eng.median()
train_eng.fillna(medians, inplace=True)
train_eng[numerical_features] = scaler.fit_transform(train_eng[numerical_features])

all_data = {
    "scaler": scaler,
    "medians": medians,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features
}

save_pickle(all_data, f'{models_path}all_data.pkl')
print("Pipline Done!")
    
train_data       = split_by_date(train_eng, dates_train)
test_data        = split_by_date(train_eng, dates_test)
print("number of dates in train = {} , number of dates in test {}".format (train_data['date_id'].nunique(),test_data['date_id'].nunique()))

cleaning = False
if cleaning:
    import gc
    #del train
    del train_eng
    gc.collect()

diff lagging 19 columns for 7 lags.
cumsum for 18 cols.
deviation 25 columns within seconds.
lagging target column for 12 lags.
Done...
we have 256 numerical and 1 categorical
Data saved to /kaggle/input/optiver-rnn-just-imb-models/all_data.pkl
Pipline Done!
number of dates in train = 481 , number of dates in test 0


In [9]:
#@title TPU
try:
    # Create a TPUClusterResolver
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    # Connect to the TPU cluster
    tf.config.experimental_connect_to_cluster(tpu)
    # Initialize the TPU system
    tf.tpu.experimental.initialize_tpu_system(tpu)
    # Create a TPUStrategy for distributed training
    tpu_strategy = tf.distribute.experimental.TPUStrategy(tpu)
except ValueError:
    tpu_strategy = None  # No TPU found

In [10]:
#@title NN second pass
#------------------------------------------------- NN Second Pass Functions -----------------------------------------------

def make_predictions(models, X_test,model = 'nn'):
    if model == 'nn':
        all_predictions = [model.predict(X_test, batch_size=16384) for model in models]
    if model == 'lgb' or model == 'xgb' or model == 'cat':
        all_predictions = [model.predict(X_test) for model in models]
    prediction = np.mean(all_predictions, axis=0)
    return prediction

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

class BestScoresCallback(Callback):
    def __init__(self):
        super().__init__()
        self.best_train_loss = float('inf')
        self.best_val_loss = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        train_loss = logs.get('loss', float('inf'))
        val_loss = logs.get('val_loss', float('inf'))

        if train_loss < self.best_train_loss:
            self.best_train_loss = train_loss
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss

    def on_train_end(self, logs=None):
        print(f"Best training loss: {self.best_train_loss}, Best validation loss: {self.best_val_loss}")


def second_pass_for_nn(df, numerical_features, categorical_features, is_inference=False):
    # Check if the DataFrame is empty
    global scaler,medians
    
    if df.empty:
        return None, None

    # Work on a copy of the DataFrame to avoid changing the original df
    df_copy = df.copy()


    # Standard scaling for numerical features
    if is_inference:
        df_copy.fillna(medians, inplace=True)
        df_copy[numerical_features] = scaler.transform(df_copy[numerical_features])

    # Preprocess Data
    df_copy['seconds_in_bucket'] = df_copy['seconds_in_bucket'] / 10
    df_copy['imbalance_buy_sell_flag'] += 1 

    # Create input vector
    X_cat = [df_copy[cat].values.reshape(-1, 1) for cat in categorical_features]
    X_num = df_copy[numerical_features].values
    

    y = df_copy['target'].values
        
    return [X_num] + X_cat, y

#------------------------------------------------- NN Second Pass Functions -----------------------------------------------

In [11]:
#@title NN model 

def create_model(categorical_features,numerical_features,initial_learning_rate=0.001):

    inputs = []
    embeddings = []

    categorical_uniques = {}
    categorical_uniques['seconds_in_bucket'] = 55

    embedding_dim = {}
    embedding_dim['seconds_in_bucket'] = 10

    input_num = Input(shape=(len(numerical_features),), name="numerical_input")
    inputs.append(input_num)

    for cat_feature in categorical_features:
        vocab_size = categorical_uniques[cat_feature]
        input_cat = Input(shape=(1,), name=f"input_{cat_feature}")
        embedding = Embedding(vocab_size, embedding_dim[cat_feature], input_length=1, name=f"embedding_{cat_feature}")(input_cat)
        flattened_embedding = Flatten(name=f"flatten_{cat_feature}")(embedding)
        inputs.append(input_cat)
        embeddings.append(flattened_embedding)

    dense_output = concatenate(embeddings + [input_num])

    dense_sizes = [512, 256, 128, 64, 32]

    for size in dense_sizes:
        dense_output = Dense(size, activation='swish')(dense_output)
        dense_output = BatchNormalization()(dense_output)
        dense_output = Dropout(0.4)(dense_output)

    output = Dense(1)(dense_output)


    lr_schedule = ExponentialDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=3000,           
    decay_rate=0.5,
    staircase=True)

    model = Model(inputs, output)
    optimizer = Adam(learning_rate=lr_schedule)

    model.compile(optimizer=optimizer, loss = "mean_absolute_error")

    return model

In [12]:
excluded_columns = ['row_id', 'date_id', 'time_id', 'target','stock_return']  

train_eng = feature_pipeline(train)
    
excluded_columns = excluded_columns + ['stock_id']
features = [col for col in train_eng.columns if col not in excluded_columns]
categorical_features =  ['seconds_in_bucket']
numerical_features = [feat for feat in features if feat not in categorical_features]
print("we have {} numerical and {} categorical".format(len(numerical_features),len(categorical_features)))


scaler = StandardScaler()
medians = train_eng.median()
train_eng.fillna(medians, inplace=True)
train_eng[numerical_features] = scaler.fit_transform(train_eng[numerical_features])

all_data = {
    "scaler": scaler,
    "medians": medians,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features
}

save_pickle(all_data, f'{models_path}all_data.pkl')
print("Pipline Done!")
    
train_data       = split_by_date(train_eng, dates_train)
test_data        = split_by_date(train_eng, dates_test)
print("number of dates in train = {} , number of dates in test {}".format (train_data['date_id'].nunique(),test_data['date_id'].nunique()))

cleaning = False
if cleaning:
    import gc
    #del train
    del train_eng
    gc.collect()

diff lagging 19 columns for 7 lags.
cumsum for 18 cols.
deviation 25 columns within seconds.
lagging target column for 12 lags.
Done...
we have 256 numerical and 1 categorical
Data saved to /kaggle/input/optiver-rnn-just-imb-models/all_data.pkl
Pipline Done!
number of dates in train = 481 , number of dates in test 0


In [13]:
#@title TPU
try:
    # Create a TPUClusterResolver
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    # Connect to the TPU cluster
    tf.config.experimental_connect_to_cluster(tpu)
    # Initialize the TPU system
    tf.tpu.experimental.initialize_tpu_system(tpu)
    # Create a TPUStrategy for distributed training
    tpu_strategy = tf.distribute.experimental.TPUStrategy(tpu)
except ValueError:
    tpu_strategy = None  # No TPU found


In [19]:
print("--------------test data is empty so adjusting the last date for test-----------------")
test_data = train_data.query("date_id > 450").copy()
train_data = train_data.query("date_id <= 450").copy()

X_train, y_train    = second_pass_for_nn(train_data,numerical_features,categorical_features)
X_test, y_test      = second_pass_for_nn(test_data,numerical_features,categorical_features)

--------------test data is empty so adjusting the last date for test-----------------


In [41]:
directory = os.path.dirname(models_path)
if not os.path.exists(directory):
    os.makedirs(directory)

callbacks = [BestScoresCallback()]  # Always include BestScoresCallback
if dates_train[1] != 480:
    early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=False)
    callbacks.append(early_stopping)

# if tpu_strategy:
# with tpu_strategy.scope():
nn_models = []
for i in range(num_models['nn']):
    print(f"Training nn model {i+1} out of {num_models['nn']} with seed {42+i}")
    print("---------------------------------------")
    set_all_seeds(42+i)
    
    nn_model = create_model(categorical_features, numerical_features, initial_learning_rate=nn_lr)
    history = nn_model.fit(X_train, y_train, validation_data=(X_test, y_test),epochs=nn_ep, batch_size=nn_bs, callbacks=callbacks)

    print("---------------------------------------")
    nn_model.save(f'{models_path}swish_model_seed_{i}.h5')
    nn_models.append(nn_model)
        
    predictions =  make_predictions(nn_models, X_test,model ='nn')                        

    print(f"Ensemble Mean Absolute Error: {mean_absolute_error(y_test, predictions):.4f}")

Training nn model 1 out of 1 with seed 42
---------------------------------------


c:\Users\cwang\anaconda3\envs\work\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 33s 200ms/step - loss: 6.4314 - val_loss: 5.8487
Epoch 2/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 29s 192ms/step - loss: 6.3388 - val_loss: 5.8372
Epoch 3/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 30s 199ms/step - loss: 6.3182 - val_loss: 5.8296
Epoch 4/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 32s 215ms/step - loss: 6.3044 - val_loss: 5.8230
Epoch 5/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 32s 215ms/step - loss: 6.2962 - val_loss: 5.8163
Epoch 6/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 31s 203ms/step - loss: 6.2901 - val_loss: 5.8136
Epoch 7/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 29s 191ms/step - loss: 6.2841 - val_loss: 5.8094
Epoch 8/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 27s 180ms/step - loss: 6.2787 - val_loss: 5.8083
Epoch 9/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 28s 186ms/step - loss: 6.2747 - val_loss: 5.8048
Epoch 10/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 28s 184ms/step - loss: 6.2714 - val_loss: 5.8024
Epoch 11/60
150/150 ━━━━━━━━━━━━━━━━━━━━ 28s 185ms/step - loss: 6.2676 - val_loss: 5.8003
Epoch 12/60
150/150

KeyboardInterrupt: 

In [60]:
nn_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_seconds_in_b… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_seconds_… │ (None, 1, 10)     │        550 │ input_seconds_in… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_seconds_in… │ (None, 10)        │          0 │ embedding_second… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numerical_input     │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 266)       │          0 │ flatten_seconds_… │
│ (Concatenate)       │                   │            │ numerical_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 512)       │    136,704 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_6[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 256)       │    131,328 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_7[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 128)       │     32,896 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_8[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 64)        │      8,256 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_9[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 32)        │      2,080 │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense_10[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 32)        │          0 │ batch_normalizat

 Total params: 315,815 (1.20 MB)

 Trainable params: 313,831 (1.20 MB)

 Non-trainable params: 1,984 (7.75 KB)

In [26]:
def clean_format(df):
    df['target'] = df['target'].astype('float64')
    for feat in ['stock_id','date_id','seconds_in_bucket']:
        df[feat] = df[feat].astype('int64')
    return df

def clean_up(df):
    df = df[['stock_id','revealed_date_id','seconds_in_bucket','revealed_target']].rename(columns={'revealed_date_id': 'date_id', 'revealed_target': 'target'})
    df['target'].fillna(-0.06020069, inplace=True)
    df = clean_format(df)
    return df

def manage_buffer(buffer, df, max_lag):

    # df['iteration'] = df['iteration'].max()  # Ensure the iteration number is consistent in the new df chunk
    if buffer.empty:
        return df.copy()
    elif buffer['iteration'].nunique() < max_lag:
        return pd.concat([buffer, df], ignore_index=True)
    else:
        oldest_iteration = buffer['iteration'].min()
        buffer = buffer[buffer['iteration'] > oldest_iteration]
        return pd.concat([buffer, df], ignore_index=True)
    

def append_target_lags(df, target_buffer, lags):

    column_to_lag = 'target'
    df_lagged = df.copy()

    for lag in lags:

        temp_df = target_buffer.copy()
        temp_df['date_id'] = temp_df['date_id'] + lag
        
        temp_df.rename(columns={column_to_lag: f'lag{lag}_{column_to_lag}'}, inplace=True)
        
        df_lagged = pd.merge(df_lagged, temp_df[['stock_id', 'date_id', 'seconds_in_bucket', f'lag{lag}_{column_to_lag}']], 
                             on=['stock_id', 'date_id', 'seconds_in_bucket'], 
                             how='left')
        
    return df_lagged


def timeseries_lag_features(data, columns_to_lag, numbers_of_lag):
    for col in columns_to_lag:
        for lag in numbers_of_lag:
            data[f'{col}_lag{lag}'] = data.groupby(['stock_id'])[col].shift(lag)
    return data


def timeseries_diff_lag_features(data, columns_to_lag, numbers_of_lag):
    # Store new columns in a dictionary before joining them to the original DataFrame
    new_columns = {}

    # Iterate over each group defined by 'stock_id' to reduce groupby operations
    grouped_data = data.groupby('stock_id')

    for col in columns_to_lag:
        for lag in numbers_of_lag:
            diff_col_name = f'{col}_diff_lag{lag}'

            # Compute the lagged difference for each group
            new_columns[diff_col_name] = data[col] - grouped_data[col].shift(lag)

    # Concatenate all new columns and join with the original DataFrame
    new_columns_df = pd.DataFrame(new_columns)
    result_df = pd.concat([data, new_columns_df], axis=1)

    return result_df


def timeseries_cumsum_features(df, columns_to_compute):
    df_copy = df.copy()
    
    # Group by 'stock_id' and 'date_id' for cumulative sum calculation
    grouped = df_copy.groupby(['stock_id'])
    
    # Calculate cumulative sum for each column within each group
    for column in columns_to_compute:
        cumsum_col_name = f'{column}_cumsum'
        df_copy[cumsum_col_name] = grouped[column].cumsum()
        
    return df_copy


def timeseries_deviation_within_seconds(df, num_features):
    # Calculating the median for each numerical feature
    medians = df[num_features].median()

    # Calculate deviations in a vectorized manner
    deviation_cols = {f'deviation_from_median_{feature}': df[feature] - medians[feature] for feature in num_features}
    df = df.assign(**deviation_cols)

    return df


import psutil

def memory_limit_exceeded(threshold_in_mb):

    process = psutil.Process()
    current_memory_usage = process.memory_info().rss / (2**20)  # Convert to MB
    return current_memory_usage > threshold_in_mb